# Phase 4 — Pipeline Complet A1 → A7 (PRD v3.0)
**YouTube Quality Analyzer** | Phase 4 — Intégration

Ce notebook exécute le pipeline LangGraph complet en version **PRD v3.0** sans CSV réel :
- Injection de commentaires synthétiques (bypass A1)
- Exécution A2 → A3‖A4‖A5 → A6 → A7
- Inspection du rapport final avec champs v3.0 (`hallucination_flags`, `fallback_used`, `sc_consensus`)

**Changements v3.0 à simuler correctement :**
| Agent | Schéma JSON mock requis |
|---|---|
| A3 | `reasoning`, `sentiment_label`, `sentiment_score` [0-1], `confidence`, `sarcasm_detected` |
| A4 | `reasoning`, `informativeness`, `argumentation`, `constructiveness`, `discourse_score` [0-1], `high_quality_indices`, `tot_used` |
| A5 | `reasoning`, `spam_ratio` [0-1], …, `noise_ratio` [0-1], `svm_used` |
| A7 | `reasoning`, `pertinence_score` [0-1], `verdict` |

> **Triple patch requis** : `agents.X.get_llm` (garde) + `models.llm_loader.get_llm` (safe_llm_call) pour chaque agent LLM.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import MagicMock, patch
from graph import run_pipeline

## 1. Commentaires synthétiques (bypass CSV)

In [ ]:
RAW_COMMENTS = [
    {"text": "This is a great tutorial on machine learning!", "video_id": "ml_vid_001"},
    {"text": "Very informative, learned a lot about gradient descent.", "video_id": "ml_vid_001"},
    {"text": "The approach mirrors Goodfellow ch.9 — excellent depth.", "video_id": "ml_vid_001"},
    {"text": "Best ML course I have taken, clearly explained backprop.", "video_id": "ml_vid_001"},
    {"text": "Could you cover transformer architectures in the next video?", "video_id": "ml_vid_001"},
    {"text": "lol 😂", "video_id": "ml_vid_001"},
    {"text": "SUBSCRIBE TO MY CHANNEL: bit.ly/spam", "video_id": "ml_vid_001"},
]

VIDEO_ID = "ml_vid_001"
TOPIC    = "machine learning"
print(f"Corpus : {len(RAW_COMMENTS)} commentaires | Vidéo : {VIDEO_ID} | Thème : {TOPIC}")

## 2. Exécution pipeline — tous LLM mockés

In [ ]:
def _make_mock_llm():
    """
    LLM mock v3.0 — retourne le bon schéma JSON selon l'agent appelant.
    Détection basée sur les champs attendus dans le prompt.
    """
    mock = MagicMock()

    def _invoke(messages, **kw):
        prompt = " ".join(
            m.content if hasattr(m, "content") else str(m)
            for m in messages
        )

        # A3 — ReAct sentiment (champs requis : reasoning + sarcasm_detected + confidence)
        if "sentiment_label" in prompt and "sarcasm" in prompt:
            body = json.dumps({
                "reasoning": "Thought 1: VADER=positive. Thought 2: no sarcasm. Thought 3: positive.",
                "sentiment_label":  "positive",
                "sentiment_score":  0.78,
                "confidence":       0.88,
                "sarcasm_detected": False,
                "rationale": "Positive corpus — quality ML content.",
            })

        # A4 CoT — discourse (champs requis : informativeness + discourse_score + tot_used)
        elif "informativeness" in prompt and "tot_used" not in prompt:
            body = json.dumps({
                "reasoning": "Thought 1: informative. Thought 2: argumentative. Thought 3: constructive.",
                "informativeness":      0.75,
                "argumentation":        0.65,
                "constructiveness":     0.80,
                "discourse_score":      0.733,
                "high_quality_indices": [0, 2, 3],
                "rationale": "Good discourse with substantive comments.",
                "tot_used":  False,
            })

        # A4 ToT — discourse (champ tot_used présent dans le prompt)
        elif "tot_used" in prompt or "Branch 1" in prompt:
            body = json.dumps({
                "reasoning": "Branch 1: deep. Branch 2: quality. Branch 3: epistemic.",
                "informativeness":      0.77,
                "argumentation":        0.68,
                "constructiveness":     0.82,
                "discourse_score":      0.757,
                "high_quality_indices": [0, 2, 3],
                "rationale": "ToT synthesis: strong discourse.",
                "tot_used":  True,
            })

        # A5 — noise CoT léger (champs requis : noise_ratio [0-1] + svm_used)
        elif "noise_ratio" in prompt or "spam_ratio" in prompt:
            body = json.dumps({
                "reasoning": "Thought 1: SVM confirms spam. Thought 2: reactions low. Thought 3: noise=0.20.",
                "spam_ratio":     0.10,
                "offtopic_ratio": 0.05,
                "reaction_ratio": 0.10,
                "toxic_ratio":    0.00,
                "bot_ratio":      0.00,
                "noise_ratio":    0.20,
                "rationale": "Low noise — mainly quality comments.",
                "svm_used":  True,
            })

        # A7 — topic matcher (champ pertinence_score)
        elif "pertinence_score" in prompt or "pertinence" in prompt.lower():
            body = json.dumps({
                "reasoning": "The comments discuss gradient descent and backprop — on-topic.",
                "pertinence_score": 0.85,
                "verdict": "The video is highly relevant to machine learning.",
            })

        # A6 — summary (texte libre, pas de JSON)
        else:
            body = (
                "The comment section reflects strong viewer engagement with high-quality "
                "technical discourse. Sentiment is overwhelmingly positive and noise levels "
                "are low, indicating a trustworthy signal."
            )

        return MagicMock(content=body)

    mock.invoke.side_effect = _invoke
    return mock


# Patches : garde de chaque agent + models.llm_loader (pour safe_llm_call)
_mock = _make_mock_llm()
patches = [
    patch("agents.a3_sentiment.get_llm",    return_value=_mock),
    patch("agents.a4_discourse.get_llm",    return_value=_mock),
    patch("agents.a5_noise.get_llm",        return_value=_mock),
    patch("agents.a6_synthesizer.get_llm",  return_value=_mock),
    patch("agents.a7_topic_matcher.get_llm",return_value=_mock),
    patch("models.llm_loader.get_llm",      return_value=_mock),  # requis pour safe_llm_call
]
for p in patches:
    p.start()

try:
    report = run_pipeline(raw_comments=RAW_COMMENTS, video_id=VIDEO_ID, topic=TOPIC)
finally:
    for p in patches:
        p.stop()

details = report.get("details", {}) or {}
print("=" * 65)
print(f"  Score_Sentiment    : {details.get('sentiment', {}).get('sentiment_score')}")
print(f"  Score_Discours     : {details.get('discourse', {}).get('discourse_score')}")
print(f"  Score_Bruit        : {details.get('noise', {}).get('noise_score')}")
print(f"  Score_Global       : {report.get('score_global')}")
print(f"  Score_Pertinence   : {report.get('score_pertinence')}")
print(f"  Score_Final        : {report.get('score_final')}")
print(f"  Quality Label      : {report.get('quality_label')}")
print(f"  Topic Verdict      : {report.get('topic_verdict')}")
print(f"  fallback_used      : {report.get('fallback_used')}")
print(f"  hallucination_flags: {report.get('hallucination_flags', [])}")
print(f"  sc_consensus       : {report.get('sc_consensus')}")
print(f"  Erreurs            : {report.get('errors')}")
print("=" * 65)

## 3. Sans topic — Score_Final = Score_Global

In [ ]:
with patch("agents.a3_sentiment.get_llm", return_value=None), \
     patch("agents.a4_discourse.get_llm",  return_value=None), \
     patch("agents.a5_noise.get_llm",      return_value=None), \
     patch("agents.a6_synthesizer.get_llm",return_value=None), \
     patch("agents.a7_topic_matcher.get_llm", return_value=None):
    report_no_topic = run_pipeline(raw_comments=RAW_COMMENTS, video_id=VIDEO_ID, topic="")

print("Sans topic :")
print(f"  Score_Global : {report_no_topic.get('score_global')}")
print(f"  Score_Final  : {report_no_topic.get('score_final')}")
assert report_no_topic["score_global"] == report_no_topic["score_final"], \
    "Sans topic, score_final doit être égal à score_global"
print("  Assertion OK : score_final == score_global quand topic est absent.")

## Résumé Pipeline Complet — PRD v3.0

| Étape | Statut |
|---|---|
| A1 → A2 → A3‖A4‖A5 → A6 → A7 (LLM mockés v3.0) | Confirmé |
| **A3** : ReAct + VADER + sarcasme | JSON v3.0 : `reasoning`, `confidence`, `sarcasm_detected` |
| **A4** : CoT + ToT conditionnel + text_stats + arg_markers | JSON v3.0 : `reasoning`, `discourse_score`, `tot_used` |
| **A5** : SVM first + CoT léger | JSON v3.0 : `reasoning`, `noise_ratio` [0-1], `svm_used` |
| **A7** : ToT systématique + Self-Consistency ×3 | JSON v3.0 : `reasoning`, `pertinence_score` [0-1] |
| `hallucination_flags` agrégés (reducer LangGraph) | Propagés de chaque agent vers le rapport final |
| `fallback_used` | `True` si au moins un agent en fallback |
| `sc_consensus` / `low_consensus` | Self-Consistency A7 — consensus ≥ 2/3 |
| Score_Global ∈ [0, 100] | Confirmé |
| Score_Final ∈ [0, 100] | Confirmé |
| Sans topic : Score_Final = Score_Global | Confirmé |
| Pipeline sans LLM (fallbacks SVM/VADER) | Confirmé |
| **Double patch** `models.llm_loader.get_llm` requis | Sinon `safe_llm_call` ignore le mock |

**Formules PRD §4.3 :**
```
Score_Global = 0.35 × Score_Sentiment + 0.40 × Score_Discours + 0.25 × Score_Bruit
Score_Final  = 0.60 × Score_Global    + 0.40 × Score_Pertinence
```